## Data cleanning and preparation

In [27]:
import pandas as pd

companies = pd.read_csv("../data/raw/linkedin-job-postings/companies/companies.csv")
salaries = pd.read_csv("../data/raw/linkedin-job-postings/jobs/salaries.csv")
postings = pd.read_csv("../data/raw/linkedin-job-postings/postings.csv")

## Handling missing values

Too many nulls in company_size in companies -> Missing values in `company_size` are replaced with an explicit "Unknown" category to preserve records while clearly indicating unavailable information.

In [2]:
companies['company_size'] = companies['company_size'].fillna('Unknown')

In [3]:
companies['company_size'].value_counts(dropna=False)

company_size
2.0        4956
1.0        4348
5.0        3918
3.0        3108
Unknown    2774
4.0        2333
7.0        1953
6.0        1083
Name: count, dtype: int64

In [4]:
companies['company_size'].isna().sum()

np.int64(0)

In [5]:
companies[companies['company_size'] == 'Unknown'].head(10)

,company_id,name,description,company_size,state,country,city,zip_code,address,url
1942,15355,The Hunter Group Associates,The Hunter Group. is a National Executive Sear...,Unknown,IL,US,Chicago,60611,0,https://www.linkedin.com/company/the-hunter-gr...
2033,16331,Strategic Solutions,NaN,Unknown,0,US,0,0,0,https://www.linkedin.com/company/strategic-sol...
3237,36325,National Crime Prevention Council,The National Crime Prevention Council’s missio...,Unknown,District of Columbia,US,Washington,20005,1100 15th St NW,https://www.linkedin.com/company/national-crim...
3470,41882,Austin Film Festival,Austin Film Festival furthers the art and craf...,Unknown,TX,US,Austin,78702,1801 Salina Street,https://www.linkedin.com/company/austin-film-f...
3721,48853,Emerald Resource Group,Emerald Resource Group is a professional staff...,Unknown,OH,US,Broadview Heights,44147,1 Eagle Valley Court,https://www.linkedin.com/company/emerald-resou...
4053,59557,Island Staffing,Island Staffing is a Corporation that Provides...,Unknown,CA,US,Oceanside,92052,1895 Avenida Del Oro,https://www.linkedin.com/company/island-staffing
4234,65296,"Request Technology, LLC","With over 39 years of experience, Request Tech...",Unknown,IL,US,Naperville,60563,200 E 5th Ave,https://www.linkedin.com/company/request-techn...
4301,68169,Discover Staffing,DISCOVER STAFFING is a business-to-business st...,Unknown,GA,US,Alpharetta,30004,12850 Highway 9 North,https://www.linkedin.com/company/discover-staf...
4545,79197,Advanced Search Group,"At Advanced Search Group, located in the great...",Unknown,IL,US,Woodridge,60517,2 Plaza Dr,https://www.linkedin.com/company/advanced-sear...
4668,84686,Pathways Personnel,Founded in 1969 Pathways Personnel uses its st...,Unknown,CA,US,San Francisco,94105,"455 Market Street,",https://www.linkedin.com/company/pathways-pers...


In [30]:
companies.isna().mean().sort_values(ascending=False)

company_size    0.113349
description     0.012136
zip_code        0.001144
state           0.000899
address         0.000899
name            0.000041
city            0.000041
company_id      0.000000
country         0.000000
url             0.000000
dtype: float64

Too many nulls in min_salary, med_salary y max_salary in salaries -> Salary information is often partially reported across `min_salary`, `med_salary`, and `max_salary`. To maximize data usability without imputing values, a unified
`salary_value` field is created by prioritizing median salary when available, followed by minimum and maximum values.

In [12]:
salaries_clean = salaries.copy()

salaries_clean['salary_value'] = (
    salaries_clean['med_salary']
    .combine_first(salaries_clean['min_salary'])
    .combine_first(salaries_clean['max_salary'])
)

In [13]:
salaries_clean['salary_value'].isna().mean()

np.float64(0.0)

In [14]:
salaries_clean[
    ['min_salary', 'med_salary', 'max_salary', 'salary_value']
].head(10)

,min_salary,med_salary,max_salary,salary_value
0,NaN,20.00,NaN,20.00
1,23.0,NaN,25.0,23.00
2,100000.0,NaN,120000.0,100000.00
3,10000.0,NaN,200000.0,10000.00
4,33.0,NaN,35.0,33.00
5,NaN,48.43,NaN,48.43
6,50000.0,NaN,80000.0,50000.00
7,84000.0,NaN,101000.0,84000.00
8,50.0,NaN,55.0,50.00
9,72000.0,NaN,100000.0,72000.00


Normalize salary_value to YEARLY

In [15]:
PAY_PERIOD_TO_YEARLY = {
    'YEARLY': 1,
    'MONTHLY': 12,
    'WEEKLY': 52,
    'BIWEEKLY': 26,
    'HOURLY': 40 * 52
}

salaries_clean = salaries_clean.copy()

salaries_clean['salary_yearly'] = (
    salaries_clean['salary_value']
    * salaries_clean['pay_period'].map(PAY_PERIOD_TO_YEARLY)
)

In [20]:
salaries_clean[
    ['salary_value', 'pay_period', 'salary_yearly']
].head(30)

,salary_value,pay_period,salary_yearly
0,20.00,HOURLY,41600.0
1,23.00,HOURLY,47840.0
2,100000.00,YEARLY,100000.0
3,10000.00,YEARLY,10000.0
4,33.00,HOURLY,68640.0
5,48.43,HOURLY,100734.4
6,50000.00,YEARLY,50000.0
7,84000.00,YEARLY,84000.0
8,50.00,HOURLY,104000.0
9,72000.00,YEARLY,72000.0


In [31]:
salaries_clean.isna().mean().sort_values(ascending=False)

med_salary           0.83234
max_salary           0.16766
min_salary           0.16766
salary_id            0.00000
job_id               0.00000
pay_period           0.00000
currency             0.00000
compensation_type    0.00000
salary_value         0.00000
salary_yearly        0.00000
dtype: float64

Too many nulls in closed_time in postings -> The `closed_time` field contains many missing values. This is expected behavior, as the field represents an event (job closing) that may not have occurred. Missing values are therefore interpreted as postings that are not explicitly marked as closed. An additional boolean field `is_closed` is created to explicitly capture the closure status of job postings.

In [28]:
postings_clean = postings.copy()

postings_clean['is_closed'] = postings_clean['closed_time'].notna()

In [29]:
postings_clean['is_closed'].value_counts()

is_closed
False    122776
True       1073
Name: count, dtype: int64

In [32]:
postings.isna().mean().sort_values(ascending=False)

closed_time                   0.991336
skills_desc                   0.980307
med_salary                    0.949293
remote_allowed                0.876898
applies                       0.811706
min_salary                    0.759441
max_salary                    0.759441
currency                      0.708734
compensation_type             0.708734
pay_period                    0.708734
normalized_salary             0.708734
posting_domain                0.322716
application_url               0.296046
formatted_experience_level    0.237459
fips                          0.221358
zip_code                      0.168528
company_name                  0.013880
company_id                    0.013864
views                         0.013638
description                   0.000057
job_id                        0.000000
title                         0.000000
location                      0.000000
original_listed_time          0.000000
formatted_work_type           0.000000
application_type         

## Fixing invalid placeholders, Column selection and standardization, Fields kept as-is and Final cleaned datasets

Registers detected with `country`, `state`, `zipcode`  field containing `"0"`, which is not valid null in companies -> Some records contained the value "0" in geographic fields (`country`, `state`, and `zip_code`), which does not represent valid information. These placeholders were replaced with proper missing values to ensure consistent data quality.

In [35]:
companies[['country', 'state', 'zip_code']].value_counts().head()

country  state       zip_code
0        0           0           715
US       0           0           597
         California  0           140
         Texas       0           110
         CA          0           109
Name: count, dtype: int64

In [34]:
companies_clean = companies.copy()

cols_with_invalid_zero = ['country', 'state', 'zip_code']

companies_clean[cols_with_invalid_zero] = (
    companies_clean[cols_with_invalid_zero]
    .replace('0', pd.NA)
)

In [36]:
companies_clean[['country', 'state', 'zip_code']].isna().mean()

country     0.029706
state       0.089772
zip_code    0.126180
dtype: float64

Registers detected with `address` field containing `"."` and `"-"`, which is not valid nul in companies -> The `address` field contained invalid placeholder values such as "." and "-", which do not represent real addresses. These values were replaced with proper missing values to ensure consistent handling of missing data.

In [37]:
companies_clean['address'] = companies_clean['address'].replace(
    {'.': pd.NA, '-': pd.NA}
)


In [38]:
companies_clean['address'].isna().mean()

np.float64(0.0013892861520859723)

In [39]:
companies_clean['address'].value_counts().head()

address
0                     3972
New York                 6
433 W Van Buren St       6
Downtown                 6
175 Greenwich St         5
Name: count, dtype: int64

In [40]:
(companies_clean['address'].isin(['.', '-'])).sum()

np.int64(0)